# Preprocessing

Pada bagian preprocessing, artikel yang telah diperoleh dari scraping pada CNBC akan dibersihkan menjadi tabel yang terstruktur dan rapih sehingga ekstraksi fitur dan representasi fitur dapat dilakukan dengan mudah. Output dari bagian ini merupakan `articles_clean.csv` yang mengandung tabel yang telah dibuat.

In [35]:
import re
import unicodedata
import pandas as pd

In [36]:
INPUT_PATH = "../data/raw/geopolitical_news.csv"
OUTDIR = "../data/processed" 
MIN_CONTENT_CHARS = 500 

## 1. Membaca data

Membaca hasil scraping `geopolitical_news.csv` dan mengkonversi kolom "published_at" menjadi tipe data datetime daripada plaintext. Row yang tidak dapat dikonversi akan dihapus karena tidak dapat diproses oleh model time-series.

In [37]:
def load_raw(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    before = len(df)
    df["published_at"] = pd.to_datetime(df["published_at"], errors="coerce")
    df = df.dropna(subset=["published_at"])
    dropped = before - len(df)
    if dropped:
        print(f"Dropped {dropped} rows with unparseable published_at")
    return df

## 2. Drop row tanpa artikel

Hasil scraping terkadang mencakup transkripsi atau caption untuk clip video, seperti `"watch now\n<title>\n05:08\nThu, May 8 2025..."`, ini tidak mengandung informasi yang berguna dan perlu dihapus.


In [38]:
VIDEO_STUB_PATTERN = re.compile(r"^\s*watch now\b", flags=re.IGNORECASE)


def is_video_stub(content: str, min_chars: int = MIN_CONTENT_CHARS) -> bool:
    return bool(VIDEO_STUB_PATTERN.match(content)) or len(content) < min_chars

## 3. Ekstraksi Dateline

Beberapa artikel mulai dengan dateline seperti `"WASHINGTON — ..."` atau `"LONDON — ..."`, ini merupakan metadata yang berguna (untuk label geografik artikelnya) maka dateline tersebut akan dipindah ke kolomnya sendiri.

In [39]:
DATELINE_RE = re.compile(r"^([A-Z][A-Za-z.\s]{2,25})[—–-]\s+")


def extract_dateline(text: str):
    m = DATELINE_RE.match(text)
    if not m:
        return None, text
    location = m.group(1).strip()
    remainder = text[m.end():]
    return location, remainder

## 4. Clean text

- Menghapus stamp durasi video seperti (`05:08`) dan stamp datetime seperti
  (`Thu, May 8 2025 2:45 PM EDT`).
- Menghapus boilerplate/noise seperti ajuan untuk subscribe, iklan, koreksi/catatan editor, dan notifikasi cookie,
- Hasil scraping mengandung banyak karakter tipografik (seperti `--`, `™`, `®`). NFKC menormalisasi karakter tersebut menjadi karakter yang identik secara visual ke dalam 1 bentuk yang konsisten
- Collapse bagian kosong dan redundant whitespace.

In [40]:
BOILERPLATE_PATTERNS = [
    r"subscribe to.*newsletter",
    r"sign up (for|here)",
    r"advertisement",
    r"cookie (policy|settings)",
    r"all rights reserved",
    r"^—?\s*cnbc\'?s .* contributed", 
    r"^read more (from|about|on)? ?cnbc",
    r"^editor\'?s note",
    r"^correction:",
]
BOILERPLATE_RE = re.compile("|".join(BOILERPLATE_PATTERNS), flags=re.IGNORECASE)

TIMESTAMP_LINE_RE = re.compile(
    r"^\s*\d{1,2}:\d{2}\s*$|"
    r"^\s*\w{3},\s+\w{3}\s+\d{1,2}\s+\d{4}.*(AM|PM).*$",
    flags=re.IGNORECASE | re.MULTILINE,
)


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = TIMESTAMP_LINE_RE.sub("", text)

    lines = [ln for ln in text.split("\n") if not BOILERPLATE_RE.search(ln.strip())]
    text = "\n".join(lines)

    text = re.sub(r"\n{2,}", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

## 5. Basic text statistics

Menghitung word count, rough sentence count, dan paragraph count untuk mendeteksi outlier dengan lebih cepat dan membantu penentuan chunking.

In [41]:
def text_stats(text: str) -> dict:
    words = re.findall(r"\w+", text)
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    sentences = [s for s in sentences if s]
    paragraphs = [p for p in text.split("\n") if p.strip()]
    return {
        "word_count": len(words),
        "sentence_count": len(sentences),
        "paragraph_count": len(paragraphs),
    }

## 6. Flag length outliers

Memberikan flag pada artikel yang terlalu panjand atau terlalu pendek menggunakan IQR bounds

In [42]:
def flag_length_outliers(df: pd.DataFrame, col: str = "word_count") -> pd.Series:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    return ~df[col].between(lower, upper)

## 7. Deduplicate

Artikel yang sama dapat discrape lebih dari sekali menggunakan URL yang berbeda ataupun URL yang tetap sama, kasus ini perlu dihapus terlebih dahulu.

In [43]:
def dedupe(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    key_title = df["clean_title"].str.strip().str.lower()
    key_content = df["clean_content"].str.strip().str.lower()
    df = df.loc[~(df["url"].duplicated() | (key_title + "||" + key_content).duplicated())]
    dropped = before - len(df)
    if dropped:
        print(f"Dropped {dropped} duplicate rows")
    return df

## 8. Menjalankan Pipeline

In [ ]:
raw = load_raw(INPUT_PATH)
print(f"Loaded {len(raw)} raw rows")

df = raw.copy()

df["is_video_stub"] = df["content"].map(is_video_stub)
n_stubs = df["is_video_stub"].sum()
df = df[~df["is_video_stub"]].drop(columns=["is_video_stub"])
print(f"Dropped {n_stubs} video-stub / near-empty rows")

dateline_and_body = df["content"].map(extract_dateline)
df["dateline_location"] = [d[0] for d in dateline_and_body]
df["content"] = [d[1] for d in dateline_and_body]

df["clean_title"] = df["title"].map(clean_text)
df["clean_content"] = df["content"].map(clean_text)
df = df[df["clean_content"].str.len() > 0]

df = dedupe(df)

stats = df["clean_content"].map(text_stats).apply(pd.Series)
df = pd.concat([df.reset_index(drop=True), stats.reset_index(drop=True)], axis=1)
df["is_length_outlier"] = flag_length_outliers(df)

df["date"] = df["published_at"].dt.date
df = df.sort_values("published_at").reset_index(drop=True)

article_df = df[[
    "published_at", "date", "clean_title", "clean_content",
    "dateline_location", "word_count", "sentence_count", "paragraph_count",
    "is_length_outlier", "source_domain", "language", "url",
]]

print(f"Final: {len(article_df)} clean articles")
print(f"{article_df['dateline_location'].notna().sum()} rows have a dateline location")
print(f"{article_df['is_length_outlier'].sum()} rows flagged as length outliers")

article_df.to_csv(f"{OUTDIR}/articles_clean.csv", index=False)

Loaded 14415 raw rows
Dropped 0 video-stub / near-empty rows
Final: 14411 clean articles
776 rows have a dateline location
601 rows flagged as length outliers


In [45]:
print(article_df[["word_count", "sentence_count", "paragraph_count"]].describe())
print(article_df["dateline_location"].value_counts().head(10))
display(article_df.head())

         word_count  sentence_count  paragraph_count
count  14411.000000     14411.00000     14411.000000
mean     727.109292        31.62147        21.764485
std      470.718199        21.96112        17.116758
min       12.000000         1.00000         1.000000
25%      440.000000        19.00000        13.000000
50%      626.000000        27.00000        18.000000
75%      892.500000        38.00000        27.000000
max    14981.000000       637.00000       873.000000
dateline_location
WASHINGTON    232
LONDON        198
BEIJING       174
DETROIT        49
SINGAPORE      34
HOUSTON        12
NEW YORK       10
SHANGHAI        4
BRUSSELS        3
CHICAGO         3
Name: count, dtype: int64


,published_at,date,clean_title,clean_content,dateline_location,word_count,sentence_count,paragraph_count,is_length_outlier,source_domain,language,url
0,2021-09-01 07:00:00,2021-09-01,"I’ve paid off almost $200,000 of student loans...","In 2015, I graduated law school with six figur...",None,1396,61,43,False,CNBC,en,https://www.cnbc.com/2021/09/01/i-paid-off-alm...
1,2021-09-01 07:00:00,2021-09-01,Weekly mortgage-refinance demand drops as inte...,A prolonged period of low mortgage rates is ta...,None,323,12,8,False,CNBC,en,https://www.cnbc.com/2021/09/01/weekly-mortgag...
2,2021-09-01 07:00:00,2021-09-01,S&P 500 and Nasdaq notch record closes as jobl...,The S&P 500 and the Nasdaq Composite climbed t...,None,394,17,9,False,CNBC,en,https://www.cnbc.com/2021/09/01/stock-market-f...
3,2021-09-01 07:00:00,2021-09-01,U.S. relationship with Taliban unclear after e...,Secretary of Defense Lloyd Austin said Wednesd...,WASHINGTON,675,37,17,False,CNBC,en,https://www.cnbc.com/2021/09/01/afghanistan-up...
4,2021-09-01 07:00:00,2021-09-01,China's health-care sector could be Beijing's ...,China's health-care sector will probably be th...,None,515,24,20,False,CNBC,en,https://www.cnbc.com/2021/09/02/chinas-regulat...
